In [14]:
from rayoptics.environment import *

l1 = open_model('LB1723-B-Zemax-ZMX.zmx')
l2 = open_model('LB1761-B-Zemax-ZMX.zmx')

sm = l1['seq_model']
sm.list_model()        # prints surfaces, radii, thicknesses, glasses
sm.list_surfaces()

              c            t        medium     mode   zdr      sd
  Obj:     0.000000  1.00000e+10       air      dummy  1      1.0000
    1:     0.016880      14.4400     N-BK7             1      25.400
    2:    -0.016880      44.2219       air             1      25.400
  Img:     0.000000      0.00000                dummy  1      3.1395
0 Surface(lbl='Obj', profile=Spherical(c=0.0), interact_mode='dummy')
1 Surface(profile=Spherical(c=0.0168804861580013), interact_mode='transmit')
2 Surface(profile=Spherical(c=-0.0168804861580013), interact_mode='transmit')
3 Surface(lbl='Img', profile=Spherical(c=0.0), interact_mode='dummy')


In [12]:
# pull verified values out of the imported single-lens models
l1_sm = l1['seq_model']
R1_L1, t_L1 = 1/l1_sm.ifcs[1].profile.cv, l1_sm.gaps[1].thi
R2_L1       = 1/l1_sm.ifcs[2].profile.cv

l2_sm = l2['seq_model']
R1_L2, t_L2 = 1/l2_sm.ifcs[1].profile.cv, l2_sm.gaps[1].thi
R2_L2       = 1/l2_sm.ifcs[2].profile.cv

# build the combined system
opm = OpticalModel()
sm  = opm['seq_model']
osp = opm['optical_spec']

osp['wvls']  = WvlSpec([(852.0, 1.0)], ref_wl=0)
osp['fov']   = FieldSpec(osp, key=['object', 'height'], value=1.5,
                         flds=[0., 1.0], is_relative=False)
osp['pupil'] = PupilSpec(osp, key=['object', 'NA'], value=0.18)

# --- build geometry first ---
sm.gaps[0].thi = 83.0
sm.add_surface([R1_L1, t_L1, 'N-BK7', 'Schott'])   # surf 1 = L1 front
sm.add_surface([R2_L1, 50.0])                       # surf 2 = L1 back
sm.add_surface([R1_L2, t_L2, 'N-BK7', 'Schott'])   # surf 3 = L2 front
sm.add_surface([R2_L2, 20.0])                       # surf 4 = L2 back

# --- now that surfaces exist, set apertures + stop ---
sm.ifcs[1].set_max_aperture(15.0)   # viewport clear radius on L1 front
sm.set_stop(1)                      # surface 1 is the aperture stop

opm.update_model()
sm.list_model()

              c            t        medium     mode   zdr      sd
  Obj:     0.000000      83.0000       air      dummy  1      1.0000
 Stop:    59.240000      14.4400     N-BK7             1      15.000
    2:   -59.240000      50.0000       air             1      1.0000
    3:    24.530000      8.98000     N-BK7             1      1.0000
    4:   -24.530000      20.0000       air             1      1.0000
  Img:     0.000000      0.00000                dummy  1      1.0000


/home/jmlevine/miniconda3/lib/python3.13/site-packages/rayoptics/parax/idealimager.py:87: RuntimeWarning: divide by zero encountered in scalar divide
  tt = -f*(m - 1)**2/m
/home/jmlevine/miniconda3/lib/python3.13/site-packages/rayoptics/parax/idealimager.py:88: RuntimeWarning: divide by zero encountered in scalar divide
  s = -f*(m - 1)/m  # = tt/(m - 1)
/home/jmlevine/miniconda3/lib/python3.13/site-packages/rayoptics/parax/idealimager.py:89: RuntimeWarning: invalid value encountered in scalar multiply
  sp = m*s
/home/jmlevine/miniconda3/lib/python3.13/site-packages/rayoptics/parax/etendue.py:179: RuntimeWarning: divide by zero encountered in scalar divide
  slpk = slp0/mag
/home/jmlevine/miniconda3/lib/python3.13/site-packages/rayoptics/parax/etendue.py:52: RuntimeWarning: divide by zero encountered in scalar divide
  epd = imager.f/fno if imager.f is not None else None


In [13]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    # 1. third-order (Seidel) aberration coefficients
    to = compute_third_order(opm)
    print(to)

    # 2. ray-fan plot
    plt.figure(FigureClass=RayFanFigure, opt_model=opm,
               data_type='Ray', scale_type=Fit.All_Same).plot()

    # 3. spot diagram at the diode
    plt.figure(FigureClass=SpotDiagramFigure, opt_model=opm,
               scale_type=Fit.All_Same).plot()
    plt.show()

              S-I          S-II         S-III      S-IV           S-V
1    2.318505e+09 -3.155557e+04  4.294810e-01  0.648102 -1.466623e-05
2    8.214234e+19  2.201918e+15  5.902492e+10  0.648102  1.582230e+06
3    5.785632e+30  1.550917e+26  4.157441e+21  0.268365  1.114458e+17
4    8.483793e+38  2.274195e+34  6.096286e+29  0.268365  1.634192e+25
sum  8.483793e+38  2.274195e+34  6.096286e+29  1.832933  1.634192e+25


ValueError: cannot reshape array of size 0 into shape (0,newaxis)